In [4]:
from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

username = maap.profile.account_info()["username"]
assert username

In [6]:
maap.register_algorithm_from_yaml_file("/projects/my-public-bucket/BirdStudy/gedi_conus_algo/algorithm_config.yaml").text

'{"code": 200, "message": {"id": "949f7499791f3718e551494666458ea4d7f219bf", "short_id": "949f7499", "created_at": "2026-10-30T17:50:17.000+00:00", "parent_ids": ["2801a65b33214d167fc0d344d863a1c7a95490fe"], "title": "Registering algorithm: gedi_rh98_conus_2km", "message": "Registering algorithm: gedi_rh98_conus_2km", "author_name": "root", "author_email": "root@356ab8adcdde", "authored_date": "2026-10-30T17:50:17.000+00:00", "committer_name": "root", "committer_email": "root@356ab8adcdde", "committed_date": "2026-10-30T17:50:17.000+00:00", "trailers": {}, "extended_trailers": {}, "web_url": "https://repo.maap-project.org/root/register-job-hysds-v4/-/commit/949f7499791f3718e551494666458ea4d7f219bf", "stats": {"additions": 6, "deletions": 6, "total": 12}, "status": "pending", "project_id": 3, "last_pipeline": {"id": 21352, "iid": 4797, "project_id": 3, "sha": "949f7499791f3718e551494666458ea4d7f219bf", "ref": "hysds-v5", "status": "pending", "source": "push", "created_at": "2026-10-30T1

In [7]:
# Cell 2
# ---------------------------------------------------------
# UPDATE THESE VARIABLES
# ---------------------------------------------------------
YOUR_MAAP_USERNAME = "abarenblitt"  # Replace with your MAAP username

ALGORITHM_NAME = "gedi_rh98_conus_2km"     # Must match the name in your yaml
ALGORITHM_VERSION = "main"                # Must match the version in your yaml

# Standard MAAP queues (Note: Check the MAAP ADE UI if you are unsure of your exact queue names)
# We want a mid-to-large queue so Dask doesn't run out of memory.
COMPUTE_QUEUE = "maap-dps-worker-32gb" 

print(f"Preparing to submit {ALGORITHM_NAME} v{ALGORITHM_VERSION} to {COMPUTE_QUEUE}...")

Preparing to submit gedi_rh98_conus_2km vmain to maap-dps-worker-32gb...


In [8]:
# Cell 3
print("Submitting job to DPS...")

try:
    # submitJob triggers the headless container execution
    job = maap.submitJob(
        identifier=f"run_conus_2km_{int(time.time())}", # A unique name for this specific run
        algo_id=ALGORITHM_NAME,
        version=ALGORITHM_VERSION,
        username=YOUR_MAAP_USERNAME,
        queue=COMPUTE_QUEUE
    )
    
    job_id = job.id
    print(f"✅ Job submitted successfully!")
    print(f"Job ID: {job_id}")

except Exception as e:
    print(f"❌ Failed to submit job: {e}")

Submitting job to DPS...
✅ Job submitted successfully!
Job ID: e08d9faf-5632-4839-ac86-0aa5639d0b77


In [9]:
# Cell 4
if 'job_id' in locals():
    print(f"Tracking status for Job ID: {job_id}\n")
    
    # States that indicate the job is finished
    completed_states = ['Succeeded', 'Failed', 'Dismissed']
    
    while True:
        try:
            status = maap.getJobStatus(job_id)
            print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Current Status: {status}")
            
            if status in completed_states:
                print(f"\nJob finished with final state: {status}")
                break
                
        except Exception as e:
            print(f"Error checking status: {e}")
            
        time.sleep(30) # Wait 30 seconds before polling again
else:
    print("No job_id found. Please run Cell 3 first.")

Tracking status for Job ID: e08d9faf-5632-4839-ac86-0aa5639d0b77

[2026-10-30 10:54:15] Current Status: Running
[2026-10-30 10:54:45] Current Status: Failed

Job finished with final state: Failed


In [ ]:
# Cell 5
if 'job_id' in locals():
    status = maap.getJobStatus(job_id)
    if status == 'Succeeded':
        print("Fetching output locations...")
        
        # Get the job results object
        result = maap.getJobResult(job_id)
        
        print("\n✅ Job Output URLs:")
        for url in result:
            print(url)
            
        print("\nNote: These files are usually automatically mounted in your MAAP workspace under:")
        print(f"/projects/my-public-bucket/dps_output/{ALGORITHM_NAME}/{ALGORITHM_VERSION}/... (Path varies by setup)")
    elif status == 'Failed':
        print("❌ Job failed. Check the DPS Job UI in the sidebar to view the error logs and trace the issue.")
    else:
        print(f"Job is not yet complete. Current status: {status}")